# 02 · Compute Metrics — M3-Score, FID, KID, SSIM

Compute evaluation metrics directly in-notebook for a real↔generated pair.
This mirrors `evaluation/eval_pipeline.py` but lets you inspect intermediates
(per-layer MMD, weights, entropy) — the parts that make M3 interpretable.

> **Tip:** Start with a small `N` (e.g. 64) on CPU to smoke-test, then bump up.

In [ ]:
import nb_config as C
C.setup()
import torch
DEVICE = C.detect_device()
N = 64          # images per split — raise to 500 for paper-grade numbers
print("device:", DEVICE, "| N:", N)

## Load images as `[N,3,224,224]` in `[0,1]`
M3 / RadioDino expect raw `[0,1]` tensors and normalize internally.

In [ ]:
import glob, os
from PIL import Image
from torchvision import transforms

_tf = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor()])

def load_m3(directory, n):
    paths = C.list_images(directory, n)
    return torch.stack([_tf(Image.open(p).convert("RGB")) for p in paths])

real = load_m3(C.REAL_DIR, N).to(DEVICE)
gen  = load_m3(C.GEN_DIR,  N).to(DEVICE)
print("real:", tuple(real.shape), "| gen:", tuple(gen.shape))

## M3-Score (the core metric)
Prunes layers via CKA on a small real subset, then scores. Returns the final
score plus per-layer distances / weights so you can see *why* it moved.

In [ ]:
from evaluation.m3_score_v2 import M3V2Metric

m3 = M3V2Metric(device=DEVICE)
m3.prune_layers_via_cka(real[:20])
res = m3(real, gen)

print(f"\nM3-Score = {res['m3_score']:.5f}")
print("active layers:", res.get("active_layers"))
import pandas as pd
def _row(d): return {f"L{k}": round(v, 4) for k, v in (d or {}).items()}
pd.DataFrame({
    "distance":   _row(res.get("layer_distances")),
    "weight":     _row(res.get("layer_weights")),
    "semanticity":_row(res.get("semanticity")),
    "stability":  _row(res.get("stability")),
    "uniqueness": _row(res.get("uniqueness")),
}).T

## FID & KID (Inception baselines)
For comparison against M3. Uses torchmetrics; needs uint8 `[0,255]` inputs.

In [ ]:
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.kid import KernelInceptionDistance

def load_uint8(directory, n, size=299):
    tf = transforms.Compose([transforms.Resize((size, size)), transforms.PILToTensor()])
    paths = C.list_images(directory, n)
    return torch.stack([tf(Image.open(p).convert("RGB")) for p in paths])

real_u = load_uint8(C.REAL_DIR, N)
gen_u  = load_uint8(C.GEN_DIR,  N)

fid = FrechetInceptionDistance(normalize=False)
fid.update(real_u, real=True); fid.update(gen_u, real=False)
print(f"FID = {float(fid.compute()):.3f}")

subset = min(50, N)
kid = KernelInceptionDistance(subset_size=subset, normalize=False)
kid.update(real_u, real=True); kid.update(gen_u, real=False)
km, ks = kid.compute()
print(f"KID = {float(km):.5f} ± {float(ks):.5f}")

## One-call full pipeline (optional)
Runs the project's official multi-metric CLI and writes a report + radar plot
to `results/notebook_eval/`. Slower but authoritative.

In [ ]:
# Uncomment to run the official pipeline end-to-end:
# C.run_script("evaluation/eval_pipeline.py", [
#     "--real_dir", C.REAL_DIR,
#     "--gen_dir",  C.GEN_DIR,
#     "--output_dir", "results/notebook_eval",
#     "--metrics", "fid", "kid", "m3", "ssim",
#     "--num_images", str(N),
#     "--device", DEVICE,
# ])